<a href="https://colab.research.google.com/github/bongsuyeong/hdac-smoking-abdominal-obesity/blob/main/Untitled25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('pyreadstat') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyreadstat'], check=True)

import numpy as np, pandas as pd
from scipy import stats
import pyreadstat

# 자료 파일 찾기: 같은 폴더, data/, 또는 Colab 업로드
CANDIDATES = ['HN24_ALL.sav', 'hn24.sav', 'data/HN24_ALL.sav', 'data/hn24.sav', '../data/HN24_ALL.sav', '../data/hn24.sav']
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        up = files.upload()                      # HN24_ALL.sav 를 선택해 올린다
        DATA_PATH = next(iter(up))
    except ImportError:
        raise FileNotFoundError('HN24_ALL.sav 를 노트북과 같은 폴더(또는 data/)에 둔다')
df, meta = pyreadstat.read_sav(DATA_PATH)
df = df.reset_index(drop=True)

# 노출: 현재흡연 (BS3_1 = 1·2 → 1, 3·8 → 0, 그 밖은 결측). 비흡연군에 과거흡연 포함
df['smoke'] = np.select([df['BS3_1'].isin([1, 2]), df['BS3_1'].isin([3, 8])], [1.0, 0.0], default=np.nan)

# 결과: 복부비만 (허리둘레 남 >=90, 여 >=85)
wc = df['HE_wc']
df['abd'] = np.where(wc.isna(), np.nan,
                     np.where(df['sex'] == 1, (wc >= 90), (wc >= 85)).astype(float))

# 공변량
df['incm_c'] = df['incm'].where(df['incm'].isin([1, 2, 3, 4]))
df['edu_c'] = df['edu'].where(df['edu'].isin([1, 2, 3, 4]))
# 최근 1년 음주: 최근 1년간 한 번이라도 마심(BD1_11 = 2~6)  <- 10주차 정의와 다르면 여기를 고친다
df['alc'] = np.select([df['BD1_11'].isin([2, 3, 4, 5, 6]), (df['BD1'] == 1) | (df['BD1_11'] == 1)],
                      [1.0, 0.0], default=np.nan)
df['pa'] = df['pa_aerobic'].where(df['pa_aerobic'].isin([0, 1]))

# 분석 대상: 만 20세 이상, 검진·설문 가중치, 허리둘레, 현재흡연이 모두 있는 사람  <- 10주차 정의와 다르면 고친다
df['in_analysis'] = ((df['age'] >= 20) & (df['wt_itvex'] > 0) & df['abd'].notna() & df['smoke'].notna()).astype(int)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
COL = dict(smoke='smoke', abd='abd', sex='sex', age='age', inc='incm_c', edu='edu_c', alc='alc', pa='pa',
           inan='in_analysis', w='wt_itvex', strata='kstrata', psu='psu')
n_all = int((df['in_analysis'] == 1).sum())
n_s = int(((df['in_analysis'] == 1) & (df['smoke'] == 1)).sum())
print(f'Table 1 분석 대상 {n_all:,}명 (현재흡연 {n_s:,} / 비흡연 {n_all - n_s:,})')
# 10주차 결과: 5,629명 (765 / 4,864)
assert (n_all, n_s) == (5629, 765), '10주차 분석 대상과 다름: in_analysis 정의를 10주차 노트북과 맞춘다'

In [ ]:
def svy_glm(y, X, w, strata, psu, family="logit", names=None, df=None):
    y = np.asarray(y, float); X = np.asarray(X, float); w = np.asarray(w, float)
    beta = np.zeros(X.shape[1])
    if family == "logit":
        p0 = np.average(y, weights=w); beta[0] = np.log(p0 / (1 - p0))
    else:
        beta[0] = np.log(np.average(y, weights=w))
    for _ in range(50):
        eta = X @ beta
        if family == "logit":
            mu = 1 / (1 + np.exp(-eta)); v = mu * (1 - mu)
        else:
            mu = np.exp(eta); v = mu
        A = X.T @ (X * (w * v)[:, None])
        step = np.linalg.solve(A, X.T @ (w * (y - mu)))
        beta += step
        if np.max(np.abs(step)) < 1e-10: break
    eta = X @ beta
    mu = 1 / (1 + np.exp(-eta)) if family == "logit" else np.exp(eta)
    v = mu * (1 - mu) if family == "logit" else mu
    A = X.T @ (X * (w * v)[:, None])
    u = X * (w * (y - mu))[:, None]
    d = pd.DataFrame(u); d["h"] = np.asarray(strata); d["c"] = np.asarray(psu)
    t = d.groupby(["h", "c"]).sum()
    nh = t.groupby(level=0).size()
    tc = t - t.groupby(level=0).transform("mean")
    adj = (nh / (nh - 1)).reindex(t.index.get_level_values(0)).values[:, None]
    B = (tc.values * adj).T @ tc.values
    Ai = np.linalg.inv(A); V = Ai @ B @ Ai
    se = np.sqrt(np.diag(V))
    if df is None: df = len(t) - t.index.get_level_values(0).nunique()
    tcrit = stats.t.ppf(0.975, df)
    out = pd.DataFrame({"coef": beta, "se": se,
                        "p": 2 * stats.t.sf(np.abs(beta / se), df)},
                       index=names or range(len(beta)))
    out["ratio"] = np.exp(out.coef)
    out["lo"] = np.exp(out.coef - tcrit * se); out["hi"] = np.exp(out.coef + tcrit * se)
    out.attrs["df"] = df
    return out

In [ ]:
def design(d, exposure, covs):
    cat = [COL[k] for k in ('sex', 'inc', 'edu') if COL[k] in covs]
    cont = [c for c in covs if c not in cat]
    X = pd.concat([d[[exposure]].astype(float),
                   pd.get_dummies(d[cat].astype('category'), drop_first=True, dtype=float) if cat else None,
                   d[cont].astype(float)], axis=1)
    X.insert(0, 'const', 1.0)
    return X

def fit(d, exposure, covs, family='logit', outcome=None, also_require=()):
    outcome = outcome or COL['abd']
    X = design(d, exposure, covs)
    # 범주형 변수의 결측은 더미 변수에서 0으로 사라지므로 따로 걸러낸다
    used = [exposure] + list(covs) + list(also_require)
    ok = ((d[COL['inan']] == 1) & d[used].notna().all(axis=1) & X.notna().all(axis=1)
          & d[outcome].notna() & d[COL['w']].notna())
    w = np.where(ok, d[COL['w']], 0.0)                      # 도메인 분석: 가중치 0
    res = svy_glm(d[outcome].fillna(0).values, X.fillna(0).values, w,
                  d[COL['strata']].values, d[COL['psu']].values, family, list(X.columns))
    n = int(ok.sum()); n_e = int((ok & (d[exposure] == 1)).sum())
    row = res.loc[exposure]
    return dict(n=n, n_exposed=n_e, n_ref=n - n_e, ratio=row.ratio, lo=row.lo, hi=row.hi,
                p=row.p, df=res.attrs['df'], table=res)

def show(name, r, label='OR'):
    print(f"{name}: n={r['n']:,} (노출 {r['n_exposed']:,} / 기준 {r['n_ref']:,})  "
          f"{label} {r['ratio']:.2f} ({r['lo']:.2f}–{r['hi']:.2f}), p={r['p']:.3f}, df={r['df']}")

In [ ]:
base = [COL['sex'], COL['age'], COL['inc'], COL['edu'], COL['alc']]
main = fit(df, COL['smoke'], base)
show('주 분석(모형 2)', main)
assert main['n'] == 5607 and round(main['ratio'], 2) == 1.02 and round(main['p'], 3) == 0.855, '11주차 결과와 다름'

In [ ]:
s1 = fit(df, COL['smoke'], base + [COL['pa']])
# 같은 사람에서의 모형 2: 신체활동 결측 사람을 가중치 0으로 두어 맞춘다(행은 지우지 않아 자유도가 같다)
main_same = fit(df, COL['smoke'], base, also_require=[COL['pa']])
show('S1 (모형 2 + 신체활동)', s1)
show('같은 사람의 모형 2', main_same)
assert s1['n'] == main_same['n']

In [ ]:
pr = fit(df, COL['smoke'], base, family='poisson')
show('S-유병률 (모형 2, 유병률비)', pr, label='PR')
show('(참고) 주 분석 오즈비', main)
assert pr['n'] == main['n']

In [ ]:
b1, b3 = df['BS1_1'], df['BS3_1']
valid = ((b1.isin([1, 2])) & (b3.isin([1, 2, 3]))) | (b1 == 3)
df['smoke_official'] = np.where(valid, ((b1 == 2) & b3.isin([1, 2])).astype(float), np.nan)

# 정의 변경으로 분류가 바뀐 사람 수 (분석 대상 안, 가중 전)
a = df[df[COL['inan']] == 1]
print(pd.crosstab(a[COL['smoke']], a['smoke_official'], dropna=False, margins=True))

s2 = fit(df, 'smoke_official', base)
show('S2 (공식 정의)', s2)

In [ ]:
rows = []
def add(name, r, ref, measure):
    rows.append({'분석': name, '효과 측도': measure, '분석 n': f"{r['n']:,}",
                 '현재흡연/기준': f"{r['n_exposed']:,}/{r['n_ref']:,}",
                 '추정값 (95% CI)': f"{r['ratio']:.2f} ({r['lo']:.2f}–{r['hi']:.2f})",
                 'p': f"{r['p']:.3f}",
                 '주 분석과의 차이': '-' if ref is None else f"{r['ratio'] / ref:.2f}배"})
add('주 분석 (모형 2)', main, None, 'OR')
add('S1: + 신체활동(교란)', s1, main_same['ratio'], 'OR')
add('  (같은 사람의 모형 2)', main_same, main['ratio'], 'OR')
add('S-유병률: 포아송 회귀', pr, main['ratio'], 'PR')
add('S2: 공식 흡연 정의', s2, main['ratio'], 'OR')
table3 = pd.DataFrame(rows)
print(table3.to_string(index=False))

In [ ]:
             분석 효과 측도  분석 n   현재흡연/기준     추정값 (95% CI)     p 주 분석과의 차이
    주 분석 (모형 2)    OR 5,607 761/4,846 1.02 (0.85–1.21) 0.855         -
 S1: + 신체활동(교란)    OR 5,255 716/4,539 1.00 (0.84–1.19) 0.978     0.98배
  (같은 사람의 모형 2)    OR 5,255 716/4,539 1.02 (0.86–1.21) 0.831     1.00배
  S-유병률: 포아송 회귀    PR 5,607 761/4,846 1.02 (0.92–1.12) 0.746     1.00배
   S2: 공식 흡연 정의    OR 5,607 754/4,853 1.02 (0.86–1.22) 0.795     1.01배